# Per-Prompt BERTScore F1 — Table 6 Fill-in

**Purpose:** Generate outputs from Best SFT (T2) and Best DPO (D3) on all 10 test prompts and compute per-prompt BERTScore F1.  
Outputs a complete markdown Table 6 ready to paste into `report.md`.

**Runtime:** T4 GPU  
**Expected time:** ~15 min total (5 min per model × 2, plus BERTScore)

In [ ]:
# Cell 1: Install dependencies (DPO-compatible versions)
!pip install -q --no-cache-dir \
    numpy==1.26.4 \
    transformers==4.44.2 \
    peft==0.12.0 \
    accelerate==0.34.2 \
    bitsandbytes==0.46.0 \
    triton==3.6.0 \
    bert-score==0.3.13

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Imports
import gc
import json
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import torch
from bert_score import score as bert_score_fn
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 4: Config
MODEL_ID          = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
SFT_T2_PATH       = "/content/drive/MyDrive/sft_trials/T2"
DPO_D3_PATH       = "/content/drive/MyDrive/dpo_trials/D3"
PROMPTS_PATH      = "/content/drive/MyDrive/Colab Notebooks/data/test_prompts.json"
MAX_NEW_TOKENS    = 200
TEMPERATURE       = 0.7
TOP_P             = 0.9

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Config set.")

In [ ]:
# Cell 5: Load prompts, references, and baseline per-prompt BERTScore F1
with open(PROMPTS_PATH) as f:
    data = json.load(f)

test_prompts    = [p["prompt"]    for p in data["prompts"]]
test_references = [p["reference"] for p in data["prompts"]]
prompt_types    = [p["type"]      for p in data["prompts"]]

# Baseline per-prompt BERTScore F1 — from baseline_results.csv
baseline_bert_f1 = [0.1038, 0.0057, 0.0389, -0.2058, -0.1355,
                    -0.1583, -0.0633, -0.2459,  0.3150,  0.1836]

print(f"Loaded {len(test_prompts)} prompts.")
for i, (t, p) in enumerate(zip(prompt_types, test_prompts)):
    print(f"  [{i+1}] {t}: {p[:60]}...")

In [ ]:
# Cell 6: Helpers
def generate_outputs(model, tokenizer, prompts):
    """Generate one response per prompt. Returns list of strings."""
    model.eval()
    tokenizer.padding_side = "left"
    generated = []
    for i, prompt in enumerate(prompts):
        formatted = (
            "Below is an instruction that describes a task. Write a response that "
            "appropriately completes the request.\n\n"
            f"### Instruction:\n{prompt}\n\n### Response:\n"
        )
        inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.inference_mode():
            out = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()
        generated.append(text)
        print(f"  [{i+1}/10] done")
    return generated


def per_prompt_bert_f1(generated, references):
    """Compute BERTScore F1 for each prompt individually. Returns list of floats."""
    scores = []
    for gen, ref in zip(generated, references):
        _, _, F1 = bert_score_fn(
            [gen], [ref], lang="en", rescale_with_baseline=True, verbose=False
        )
        scores.append(round(F1[0].item(), 4))
    return scores


print("Helpers defined.")

In [ ]:
# Cell 7: Evaluate Best SFT — T2
print("Loading T2 (SFT)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
sft_model = PeftModel.from_pretrained(base, SFT_T2_PATH)

print("Generating T2 outputs...")
sft_generated = generate_outputs(sft_model, tokenizer, test_prompts)

print("\nComputing per-prompt BERTScore F1 for T2...")
sft_bert_f1 = per_prompt_bert_f1(sft_generated, test_references)
print("T2 per-prompt BERTScore F1:", sft_bert_f1)
print(f"T2 mean: {sum(sft_bert_f1)/len(sft_bert_f1):.4f}")

del sft_model, base
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 8: Evaluate Best DPO — D3
# Load stack: base model → SFT T2 adapter → DPO D3 adapter
print("Loading D3 (DPO on top of T2 SFT)...")
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
sft_layer = PeftModel.from_pretrained(base, SFT_T2_PATH)
dpo_model  = PeftModel.from_pretrained(sft_layer, DPO_D3_PATH)

print("Generating D3 outputs...")
dpo_generated = generate_outputs(dpo_model, tokenizer, test_prompts)

print("\nComputing per-prompt BERTScore F1 for D3...")
dpo_bert_f1 = per_prompt_bert_f1(dpo_generated, test_references)
print("D3 per-prompt BERTScore F1:", dpo_bert_f1)
print(f"D3 mean: {sum(dpo_bert_f1)/len(dpo_bert_f1):.4f}")

del dpo_model, sft_layer, base
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 9: Print complete Table 6 — copy this output into report.md Section 6.1
prompt_ids   = list(range(1, 11))
type_labels  = ["Factual", "Factual", "Instruction", "Instruction",
                "Instruction", "Math reasoning", "Logic",
                "Creative", "Summarization", "Open-ended"]

header = "| ID | Type | Base BERTScore F1 | Best SFT (T2) | Best DPO (D3) |"
sep    = "|----|------|-------------------|---------------|---------------|"
rows   = []
for i in range(10):
    rows.append(
        f"| {i+1} | {type_labels[i]} "
        f"| {baseline_bert_f1[i]:.4f} "
        f"| {sft_bert_f1[i]:.4f} "
        f"| {dpo_bert_f1[i]:.4f} |"
    )

base_mean = sum(baseline_bert_f1) / len(baseline_bert_f1)
sft_mean  = sum(sft_bert_f1)      / len(sft_bert_f1)
dpo_mean  = sum(dpo_bert_f1)      / len(dpo_bert_f1)
mean_row  = f"| **Mean** | | **{base_mean:.4f}** | **{sft_mean:.4f}** | **{dpo_mean:.4f}** |"

print("\n" + "="*70)
print("TABLE 6 — paste into report.md Section 6.1")
print("="*70)
print(header)
print(sep)
for row in rows:
    print(row)
print(mean_row)
print("="*70)

In [ ]:
# Cell 10 (optional): Print generated texts for spot-checking
print("Generated outputs — T2 vs D3:\n")
for i in range(10):
    print(f"[{i+1}] {test_prompts[i][:70]}")
    print(f"  T2  : {sft_generated[i][:120]}")
    print(f"  D3  : {dpo_generated[i][:120]}")
    print(f"  Ref : {test_references[i][:120]}")
    print(f"  BERT F1 → Base: {baseline_bert_f1[i]:.4f}  SFT: {sft_bert_f1[i]:.4f}  DPO: {dpo_bert_f1[i]:.4f}")
    print()